In [1]:
import pandas as pd
import numpy as np
import pickle
from pulp import *


In [2]:
# Load the trained ML models we saved on Day 3
with open(r"C:\Users\dhruv\playXI\src\batsman_model.pkl", 'rb') as f:
    batsman_model = pickle.load(f)

with open(r"C:\Users\dhruv\playXI\src\bowler_model.pkl", 'rb') as f:
    bowler_model = pickle.load(f)

# Load the stats data
batsman_stats = pd.read_csv(r"C:\Users\dhruv\playXI\data\batsman_stats.csv")
bowler_stats = pd.read_csv(r"C:\Users\dhruv\playXI\data\bowler_stats.csv")

print("Models and data loaded successfully!")

Models and data loaded successfully!


In [3]:
# Calculate recent form for all players
# Sort by matchId so matches are in chronological order
batsman_stats = batsman_stats.sort_values('matchId')
bowler_stats = bowler_stats.sort_values('matchId')

# Calculate fantasy points for batsmen (same formula as Day 3)
def calculate_batting_points(row):
    points = 0
    points += row['total_runs']
    if row['total_runs'] >= 50:
        points += 8
    if row['total_runs'] >= 100:
        points += 16
    if row['balls_faced'] >= 10:
        if row['strike_rate'] >= 170:
            points += 6
        elif row['strike_rate'] >= 150:
            points += 4
        elif row['strike_rate'] >= 130:
            points += 2
    return points

def calculate_bowling_points(row):
    points = 0
    points += row['wickets'] * 25
    if row['wickets'] >= 3:
        points += 4
    if row['wickets'] >= 4:
        points += 8
    if row['balls_bowled'] >= 12:
        if row['economy'] <= 5:
            points += 6
        elif row['economy'] <= 6:
            points += 4
        elif row['economy'] <= 7:
            points += 2
    return points

batsman_stats['fantasy_points'] = batsman_stats.apply(calculate_batting_points, axis=1)
bowler_stats['fantasy_points'] = bowler_stats.apply(calculate_bowling_points, axis=1)

# Calculate recent form
batsman_stats['recent_form'] = (
    batsman_stats.groupby('batsman')['fantasy_points']
    .transform(lambda x: x.shift(1).rolling(window=5, min_periods=1).mean())
)
batsman_stats['recent_form'] = batsman_stats['recent_form'].fillna(batsman_stats['fantasy_points'].mean())

bowler_stats['recent_form'] = (
    bowler_stats.groupby('bowler')['fantasy_points']
    .transform(lambda x: x.shift(1).rolling(window=5, min_periods=1).mean())
)
bowler_stats['recent_form'] = bowler_stats['recent_form'].fillna(bowler_stats['fantasy_points'].mean())

print("Recent form calculated!")

Recent form calculated!


In [4]:
# Get the most recent form for each player
# This is what we'll use to predict their next match performance

latest_batsman_form = batsman_stats.groupby('batsman')['recent_form'].last().reset_index()
latest_batsman_form.columns = ['player', 'recent_form']
latest_batsman_form['role'] = 'batsman'

latest_bowler_form = bowler_stats.groupby('bowler')['recent_form'].last().reset_index()
latest_bowler_form.columns = ['player', 'recent_form']
latest_bowler_form['role'] = 'bowler'

print("Latest batsman form:")
print(latest_batsman_form.head())
print("\nLatest bowler form:")
print(latest_bowler_form.head())

Latest batsman form:
           player  recent_form     role
0  A Ashish Reddy         19.2  batsman
1        A Badoni         50.2  batsman
2      A Chandila          0.0  batsman
3        A Chopra          8.4  batsman
4     A Choudhary         10.5  batsman

Latest bowler form:
           player  recent_form    role
0  A Ashish Reddy    15.400000  bowler
1        A Badoni    15.000000  bowler
2      A Chandila    17.800000  bowler
3     A Choudhary    26.500000  bowler
4     A Dananjaya    24.763236  bowler


In [5]:
# Get the most recent form for each player
latest_batsman_form = batsman_stats.groupby('batsman')['recent_form'].last().reset_index()
latest_batsman_form.columns = ['player', 'recent_form']
latest_batsman_form['role'] = 'batsman'

latest_bowler_form = bowler_stats.groupby('bowler')['recent_form'].last().reset_index()
latest_bowler_form.columns = ['player', 'recent_form']
latest_bowler_form['role'] = 'bowler'

print("Latest batsman form:")
print(latest_batsman_form.head())
print("\nLatest bowler form:")
print(latest_bowler_form.head())

Latest batsman form:
           player  recent_form     role
0  A Ashish Reddy         19.2  batsman
1        A Badoni         50.2  batsman
2      A Chandila          0.0  batsman
3        A Chopra          8.4  batsman
4     A Choudhary         10.5  batsman

Latest bowler form:
           player  recent_form    role
0  A Ashish Reddy    15.400000  bowler
1        A Badoni    15.000000  bowler
2      A Chandila    17.800000  bowler
3     A Choudhary    26.500000  bowler
4     A Dananjaya    24.763236  bowler


In [6]:
# Combine batsmen and bowlers into one player pool
all_players = pd.concat([latest_batsman_form, latest_bowler_form], ignore_index=True)

# Predict fantasy points using our ML models
batsman_predictions = batsman_model.predict(
    latest_batsman_form[['recent_form']]
)
bowler_predictions = bowler_model.predict(
    latest_bowler_form[['recent_form']]
)

latest_batsman_form['predicted_points'] = batsman_predictions
latest_bowler_form['predicted_points'] = bowler_predictions

# Combine into one table
all_players = pd.concat([latest_batsman_form, latest_bowler_form], ignore_index=True)

# Add dummy credits (in real app this comes from Dream11)
all_players['credits'] = np.where(
    all_players['predicted_points'] > 50, 9,
    np.where(all_players['predicted_points'] > 30, 8, 7)
)

print("All players with predicted points:")
print(all_players.head(10))
print("\nTotal players:", len(all_players))

All players with predicted points:
           player  recent_form     role  predicted_points  credits
0  A Ashish Reddy    19.200000  batsman         23.517363        7
1        A Badoni    50.200000  batsman         24.703535        7
2      A Chandila     0.000000  batsman          7.989177        7
3        A Chopra     8.400000  batsman         11.129595        7
4     A Choudhary    10.500000  batsman          6.874650        7
5     A Dananjaya    22.262406  batsman         12.493582        7
6      A Flintoff    25.000000  batsman         24.314782        7
7        A Kamboj     2.200000  batsman          6.322479        7
8        A Kumble     1.200000  batsman          5.282383        7
9       A Manohar    11.400000  batsman         15.459184        7

Total players: 1255


In [7]:
# Build the Dream11 Team Optimizer using PuLP
def optimize_team(players_df, total_credits=100, team_size=11):
    
    # Create the optimization problem
    prob = LpProblem("Dream11_Team_Optimizer", LpMaximize)
    
    # Create a binary variable for each player (1 = selected, 0 = not selected)
    player_vars = LpVariable.dicts("player", players_df.index, cat='Binary')
    
    # Objective: Maximize total predicted points
    prob += lpSum([players_df.loc[i, 'predicted_points'] * player_vars[i] 
                   for i in players_df.index])
    
    # Constraint 1: Exactly 11 players
    prob += lpSum([player_vars[i] for i in players_df.index]) == team_size
    
    # Constraint 2: Total credits <= 100
    prob += lpSum([players_df.loc[i, 'credits'] * player_vars[i] 
                   for i in players_df.index]) <= total_credits
    
    # Constraint 3: At least 3 batsmen
    batsmen_idx = players_df[players_df['role'] == 'batsman'].index
    prob += lpSum([player_vars[i] for i in batsmen_idx]) >= 3
    
    # Constraint 4: At least 3 bowlers
    bowlers_idx = players_df[players_df['role'] == 'bowler'].index
    prob += lpSum([player_vars[i] for i in bowlers_idx]) >= 3
    
    # Solve the problem
    prob.solve(PULP_CBC_CMD(msg=0))
    
    # Get selected players
    selected = [players_df.loc[i, 'player'] 
                for i in players_df.index if player_vars[i].value() == 1]
    
    return selected

# Run optimizer
selected_team = optimize_team(all_players)

print("Your Optimized Dream11 Team:")
print("="*40)
for i, player in enumerate(selected_team, 1):
    player_info = all_players[all_players['player'] == player].iloc[0]
    print(f"{i}. {player} ({player_info['role']}) - {player_info['predicted_points']:.1f} pts")

Your Optimized Dream11 Team:
1. AP Majumdar (batsman) - 39.6 pts
2. HM Amla (batsman) - 41.5 pts
3. PP Shaw (batsman) - 39.0 pts
4. SA Yadav (batsman) - 42.9 pts
5. Shubman Gill (batsman) - 46.1 pts
6. Sikandar Raza (batsman) - 39.0 pts
7. A Flintoff (batsman) - 24.3 pts
8. JE Taylor (batsman) - 10.1 pts
9. JR Hazlewood (batsman) - 4.2 pts
10. KMDN Kulasekara (batsman) - 12.5 pts
11. SM Harwood (batsman) - 6.9 pts


In [8]:
# Fix player roles properly
# Players who appear in both tables are all-rounders
batsman_set = set(latest_batsman_form['player'])
bowler_set = set(latest_bowler_form['player'])
allrounder_set = batsman_set.intersection(bowler_set)

# Reassign roles
def assign_role(player, batsman_set, bowler_set, allrounder_set):
    if player in allrounder_set:
        return 'allrounder'
    elif player in batsman_set:
        return 'batsman'
    else:
        return 'bowler'

all_players['role'] = all_players['player'].apply(
    lambda x: assign_role(x, batsman_set, bowler_set, allrounder_set)
)

# Remove duplicates - keep highest predicted points for each player
all_players = all_players.sort_values('predicted_points', ascending=False)
all_players = all_players.drop_duplicates(subset='player', keep='first')
all_players = all_players.reset_index(drop=True)

print("Role distribution:")
print(all_players['role'].value_counts())
print("\nTotal unique players:", len(all_players))

Role distribution:
role
allrounder    487
batsman       217
bowler         64
Name: count, dtype: int64

Total unique players: 768


In [9]:
# Run optimizer again with fixed roles
def optimize_team_v2(players_df, total_credits=100, team_size=11):
    
    prob = LpProblem("Dream11_Team_Optimizer_v2", LpMaximize)
    player_vars = LpVariable.dicts("player", players_df.index, cat='Binary')
    
    # Objective: Maximize total predicted points
    prob += lpSum([players_df.loc[i, 'predicted_points'] * player_vars[i] 
                   for i in players_df.index])
    
    # Constraint 1: Exactly 11 players
    prob += lpSum([player_vars[i] for i in players_df.index]) == team_size
    
    # Constraint 2: Total credits <= 100
    prob += lpSum([players_df.loc[i, 'credits'] * player_vars[i] 
                   for i in players_df.index]) <= total_credits
    
    # Constraint 3: At least 3 batsmen (batsmen + allrounders)
    bat_idx = players_df[players_df['role'].isin(['batsman', 'allrounder'])].index
    prob += lpSum([player_vars[i] for i in bat_idx]) >= 3
    
    # Constraint 4: At least 3 bowlers (bowlers + allrounders)
    bowl_idx = players_df[players_df['role'].isin(['bowler', 'allrounder'])].index
    prob += lpSum([player_vars[i] for i in bowl_idx]) >= 3
    
    # Constraint 5: At least 1 pure bowler
    pure_bowl_idx = players_df[players_df['role'] == 'bowler'].index
    prob += lpSum([player_vars[i] for i in pure_bowl_idx]) >= 1
    
    # Solve
    prob.solve(PULP_CBC_CMD(msg=0))
    
    selected = [players_df.loc[i, 'player'] 
                for i in players_df.index if player_vars[i].value() == 1]
    return selected

# Run it
selected_team = optimize_team_v2(all_players)

print("Your Optimized PlayXI Dream11 Team:")
print("="*45)
for i, player in enumerate(selected_team, 1):
    info = all_players[all_players['player'] == player].iloc[0]
    print(f"{i}. {player} ({info['role']}) - {info['predicted_points']:.1f} pts")

total = sum(all_players[all_players['player'] == p]['predicted_points'].values[0] 
            for p in selected_team)
print(f"\nTotal Predicted Points: {total:.1f}")

Your Optimized PlayXI Dream11 Team:
1. SM Harwood (allrounder) - 48.4 pts
2. Shubman Gill (batsman) - 46.1 pts
3. JE Taylor (allrounder) - 45.9 pts
4. KMDN Kulasekara (allrounder) - 45.9 pts
5. A Flintoff (allrounder) - 45.9 pts
6. JR Hazlewood (allrounder) - 45.0 pts
7. SA Yadav (allrounder) - 42.9 pts
8. HM Amla (batsman) - 41.5 pts
9. AP Majumdar (batsman) - 39.6 pts
10. PP Shaw (batsman) - 39.0 pts
11. MP Yadav (bowler) - 36.5 pts

Total Predicted Points: 476.7


In [10]:
# Save the final player data for use in Streamlit app
all_players.to_csv(r"C:\Users\dhruv\playXI\data\all_players.csv", index=False)
print("Player data saved!")

Player data saved!
